[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Time_Series/Intro_AdFilt_APA.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Adaptive Filtering: LMS to the Affine Projection Algorithm

In [Filter Design](../Intro_DSP/Filter_Design.ipynb) we designed filters *once*, by hand. But what if the interference drifts, the room echo changes, the channel fades? Then the filter must **redesign itself from the data, every sample**. This workshop builds that idea from steepest descent → LMS → NLMS → the Affine Projection Algorithm (APA).

## 0. Introduction

The canonical setup — *system identification*:

```
 x[n] ──► unknown system ──► d[n] (+ noise)
 x[n] ──► our filter w ────► y[n]
                 e[n] = d[n] − y[n]  drives the adaptation
```

The same loop, rewired, does echo cancellation, channel equalization, and noise cancellation — adaptive filtering is one algorithm wearing four costumes.

## 1. Pre-requisites

- [Foundations of Signal Processing](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) — convolution, FIR structure.
- [Intro to Python](../Intro_Programming/Intro_Python/Intro_Python.ipynb) — NumPy.
- Comfort with gradients (any calculus course).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

# The "unknown" system our filter must discover: a 16-tap FIR echo path
M = 16
w_true = np.exp(-0.4 * np.arange(M)) * np.cos(0.9 * np.arange(M))
w_true /= np.linalg.norm(w_true)

N = 4000                       # samples
x = rng.standard_normal(N)     # white input (we'll break this assumption later!)
d = np.convolve(x, w_true)[:N] + 0.01 * rng.standard_normal(N)

plt.figure(figsize=(7, 2.2))
plt.stem(w_true)
plt.title("The unknown system's impulse response (our target)")
plt.tight_layout(); plt.show()

**What just happened.** The target: a 16-tap impulse response that oscillates while decaying — an exponentially-damped cosine, normalised to unit norm. This is what every algorithm in the workshop is trying to discover, and it is worth looking at before any of them run, because "did it converge?" is really "do these 16 stems match?"

The shape is chosen to be realistic rather than convenient. A decaying oscillation is what an acoustic echo path or a multipath radio channel actually looks like: energy arrives, reflects, and dies away. And normalising to unit norm fixes the scale so that a weight error of 0.0042 later on can be read directly as a fraction — roughly 0.4% of the system's total magnitude.

Note also the input: `x = rng.standard_normal(N)`, white noise. The comment flags that we will break this assumption later, and that flag matters. White input makes the autocorrelation matrix $R$ a scaled identity, which is the friendliest possible case for every gradient method — a perfectly circular error bowl. Session 2 replaces it with correlated input, the bowl becomes an elongated valley, and the ranking of the algorithms changes completely. Everything in Session 1 is measured under easy conditions, deliberately.

---
### 🕐 Session 1 of 2 — *From Steepest Descent to LMS* (~35 min)
**Goal:** derive the Wiener solution, then strip it down to the LMS update and watch it converge.
**Feeds into:** Session 2 (NLMS & APA).

---

## 2. Theory: the Wiener Filter & LMS

💡 **Intuition.** Picture the error surface: for an FIR filter the mean-squared error is a **bowl** in weight space — one unique bottom (the Wiener solution). Steepest descent walks downhill using the *true* gradient, which needs statistics ($R$, $p$) we never have. LMS makes one audacious move: replace the expected gradient with its **one-sample estimate**. Each step is noisy, but the *average* direction is still downhill.

### 2.1. The Wiener Solution

Minimizing $J(\mathbf{w}) = E[e^2[n]]$ with $e[n] = d[n] - \mathbf{w}^T\mathbf{x}[n]$ gives

$$\nabla J = 2R\mathbf{w} - 2\mathbf{p} = 0 \;\Rightarrow\; \mathbf{w}_o = R^{-1}\mathbf{p}$$

with $R = E[\mathbf{x}\mathbf{x}^T]$ (input autocorrelation) and $\mathbf{p} = E[d\,\mathbf{x}]$ (cross-correlation). Two problems: we don't know $R$ and $\mathbf{p}$, and inverting $R$ is expensive. Adaptive filtering is the art of *approaching* $\mathbf{w}_o$ without ever forming it.

### 2.2. The LMS Update

Substitute the instantaneous gradient estimate $\hat{\nabla} J = -2 e[n]\, \mathbf{x}[n]$ into gradient descent:

$$\boxed{\;\mathbf{w}[n+1] = \mathbf{w}[n] + \mu\, e[n]\, \mathbf{x}[n]\;}$$

Three multiplies per tap per sample. Convergence requires $0 < \mu < \frac{2}{\lambda_{max}}$ (the largest eigenvalue of $R$) — step too far and the bowl becomes a trampoline.

In [ ]:

# YOUR CODE HERE


**What just happened.** Final weight error **0.0042**. Because `w_true` was normalised to unit norm, that reads directly as a percentage: LMS recovered the unknown system to within about **0.4%**, using nothing but the input, the noisy output, and three multiplies per tap per sample. It never formed $R$, never estimated $\mathbf{p}$, and never inverted anything.

That is worth pausing on. The Wiener solution $R^{-1}\mathbf{p}$ is the exact optimum and requires statistics we do not have plus an $O(M^3)$ inversion. LMS approaches it with an update whose gradient estimate is, on any individual sample, badly wrong — a single noisy product $e[n]\mathbf{x}[n]$ standing in for an expectation. Each step is close to random; only the *average* direction is downhill. Over 4000 samples that averaging is enough.

**Why it does not reach zero.** Two separate reasons, and students routinely merge them. First, `d` carries additive noise of standard deviation 0.01 that no filter can predict — an irreducible floor. Second, and more interesting, LMS settles slightly *above* even that floor: the gradient noise keeps jittering the weights around the optimum rather than parking on it. That excess is **misadjustment**, and it scales with $\mu$. Larger steps converge faster and sit further from the bottom, which is the central trade of the whole field and the reason the comparison table in Session 2 has a misadjustment row.

**And remember the conditions.** The input here is white, so $R$ is nearly a scaled identity and the error surface is a near-circular bowl — the easiest geometry gradient descent can be handed. Try `mu=0.2` to watch the stability bound $\mu < 2/\lambda_{\max}$ fail loudly. Then keep 0.0042 in mind for Session 2, where the input becomes correlated, the bowl stretches into a narrow valley, and plain gradient methods start to struggle badly.

In [ ]:

# YOUR CODE HERE


The error drops ~30+ dB and flattens at the noise floor: the filter has *discovered* the unknown system. Try `mu=0.2` — divergence is loud and immediate.

---
### 🕐 Session 2 of 2 — *NLMS & the Affine Projection Algorithm* (~40 min)
**Goal:** fix LMS's sensitivity to input power and correlation; implement NLMS and APA and compare all three.
**Builds on:** Session 1.

---

## 3. NLMS: Normalize the Step

💡 **Intuition.** LMS's step size is entangled with the input's *power*: loud input ⇒ effectively huge steps ⇒ instability. NLMS divides the step by $\|\mathbf{x}[n]\|^2$, so each update moves the weights *just enough to cancel the current error* — a projection onto the hyperplane of solutions for the newest sample, scaled by $\mu$.

$$\mathbf{w}[n+1] = \mathbf{w}[n] + \frac{\mu}{\epsilon + \|\mathbf{x}[n]\|^2}\, e[n]\, \mathbf{x}[n]$$

with small $\epsilon$ guarding the division. Stable for $0 < \mu < 2$ regardless of input power.

In [ ]:

# YOUR CODE HERE


## 4. APA: Project Onto Many Constraints at Once

💡 **Intuition.** NLMS satisfies only the **newest** sample's equation, so with *correlated* input (speech, music — anything non-white) consecutive updates keep undoing each other, and convergence crawls. APA keeps the last $K$ input vectors and jumps to the nearest weight vector satisfying **all $K$ equations simultaneously** — an affine projection. $K=1$ recovers NLMS; larger $K$ churns through correlated input dramatically faster, at the price of a $K \times K$ solve per sample.

With $X_K[n] = [\mathbf{x}[n], \dots, \mathbf{x}[n-K+1]]$ (an $M \times K$ matrix) and $\mathbf{e}_K[n]$ the corresponding error vector:

$$\mathbf{w}[n+1] = \mathbf{w}[n] + \mu\, X_K (X_K^T X_K + \epsilon I)^{-1} \mathbf{e}_K[n]$$

In [ ]:

# YOUR CODE HERE


### 4.1. The Fair Fight: Correlated Input

White input flatters every algorithm. Real signals are colored — so we generate a correlated input by low-pass filtering noise (an AR(1) process) and race all three.

In [ ]:

# YOUR CODE HERE


**What just happened.** Same $\mu$, same data, same target — and each increase in projection order $K$ steepens the initial descent. NLMS ($K=1$) crawls, APA-4 is faster, APA-8 faster still. The input is `lfilter([1.0], [1.0, -0.9], ...)`, a strongly resonant one-pole process, so consecutive samples are heavily correlated and this is the case NLMS handles worst.

**The geometry explains the ordering.** NLMS projects onto the hyperplane of weights consistent with the *newest* sample only. When consecutive input vectors point in nearly the same direction — which is what correlated input means — each projection substantially undoes the previous one, and the weights zig-zag across a narrow valley instead of travelling along it. APA projects onto the intersection of the last $K$ hyperplanes at once, so the update is consistent with $K$ samples simultaneously and cannot be immediately undone by the next. More constraints per step, less thrashing.

This is the same conditioning story that runs through the whole curriculum: correlated input means an ill-conditioned $R$, gradient methods slow down in proportion to its condition number, and the cure is to use more local geometry per step. APA is a partial dose; [RLS](./Intro_RLS.ipynb) is the full one, carrying $R^{-1}$ outright.

**Now read the printed number carefully, because it does not say what it appears to.** APA $K=8$ ends with weight error **0.0126**, which is *larger* than Session 1's LMS result of 0.0042. That is not APA losing. The two runs use different inputs — white in Session 1, heavily correlated here — so the comparison is not apples to apples, and this notebook never measures NLMS and APA on identical footing at convergence.

What the number does illustrate is the table's last row: **misadjustment grows with $K$**. Averaging over more constraints per update amplifies the gradient noise, so the weights jitter more widely around the optimum once they arrive. APA buys convergence *speed* and pays in steady-state *precision* — the same trade LMS makes through $\mu$, now made through $K$.

The demo does not isolate that claim, and it would be a good exercise to. Run all three on the white input from Session 1 and the speed advantage largely disappears, since there is no correlation left to exploit; run them all far past convergence on the correlated input and the misadjustment ordering becomes measurable. Both variants are a few lines, and each tests one claim in the table rather than leaving the plot to imply everything at once.

Read the plot: same $\mu$, same data — but each increase in $K$ steepens the initial descent. The trade-offs to remember:

| | LMS | NLMS | APA-$K$ |
|---|---|---|---|
| Cost / sample | $O(M)$ | $O(M)$ | $O(K^2 M)$ |
| Robust to input power | ✗ | ✓ | ✓ |
| Fast on colored input | ✗ | ✗ | ✓ |
| Misadjustment (noise amp.) | low | med | grows with $K$ |

## 5. Conclusion

One loop — predict, err, correct — with increasingly clever corrections: LMS follows a noisy gradient, NLMS normalizes it, APA projects onto a window of constraints. The next step up that ladder replaces "window of constraints" with a full statistical model of the system's evolution — that is the Kalman filter.

---
## Where next

- [Adaptive Filtering: Kalman](./Intro_AdFilt_KF.ipynb) — the optimal recursive estimator.
- [Recurrent Neural Networks](./README.md#workshop-3--recurrent-neural-networks-available) — nonlinear models with internal state, same predict/correct heartbeat.
- [Filter Design](../Intro_DSP/Filter_Design.ipynb) — the fixed-filter baseline all of this improves upon.